# CONNECT TO running server as client

In [1]:
# server has to run first (buildandrun powershell)

In [2]:
# OLD CODE - keep in case
# this code lets you send messages to the server but can not receive from server to jupyter client 
  
# import socketio
# sio = socketio.Client()
# sio.disconnect()

# uid = 'jupyter-client'

# @sio.event(namespace='/main')
# def connect():
#     print("✅ Connected to /main")


# # Variables to store received data
# latest_data = None
# message_log = []

# # jupyter client listener for server events 
# @sio.on('ex', namespace='/main')
# def on_ex(data):
#     global latest_data, message_log
#     print(f"📩 Server returned on 'ex': {data}")
#     latest_data = data
#     message_log.append({'event': 'ex', 'data': data}

#     # Example: auto-react
#     if data.get('fn') == 'dropdown' and data.get('id') == 'projDD':
#         print("🧠 Project updated:", data.get('val'))
 
# # establish a connection to the server as another client 
# sio.connect('http://127.0.0.1:5000', namespaces=['/main'])
# sio.emit('join', {'usr': uid}, namespace='/main')
# sio.emit('init-project', namespace='/main')
# -----------------------------------------------------------------------------------------

import socketio
import time
import threading

# Setup Socket.IO client
sio = socketio.Client()
uid = 'jupyter-client'

latest_data = None
message_log = []

# 1️⃣ Define listeners BEFORE connect
@sio.on('ex', namespace='/main')
def on_ex(data):
    global latest_data, message_log
    print(f"📩 Server returned on 'ex': {data}")
    latest_data = data
    message_log.append({'event': 'ex', 'data': data})

@sio.event(namespace='/main')
def connect():
    print("✅ Connected to /main")
    sio.emit('join', {'usr': uid}, namespace='/main')

# 2️⃣ Connect
sio.connect('http://127.0.0.1:5000', namespaces=['/main'])

# 3️⃣ Start background thread AFTER connect
def wait_forever():
    while True:
        time.sleep(1)

thread = threading.Thread(target=wait_forever)
thread.daemon = True
thread.start()

print("📡 Listening for messages in the background...")


📡 Listening for messages in the background...
✅ Connected to /main


📩 Server returned on 'ex': {'usr': 'jupyter-client', 'val': {'name': 'CircLadderGraph-xsmall', 'layouts': ['layout1-spring', 'layout2-spring', 'layout3-spring', 'layout4-clusters'], 'layoutsRGB': ['layout1-spring', 'layout2-spring', 'layout3-spring', 'layout4-clusters'], 'links': ['layout1-spring', 'layout2-spring', 'layout3-spring', 'layout4-clusters'], 'linksRGB': ['layout1-spring', 'layout2-spring', 'layout3-spring', 'layout4-clusters'], 'selections': [{'name': 'cluster group 3', 'nodes': ['134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159', '160', '161', '162', '163', '164', '165', '166', '167', '168', '169', '170', '171', '172', '173', '174', '175', '176', '177', '178', '179', '180', '181', '182', '183', '184', '185', '186', '187', '188', '189', '190', '191', '192', '193', '194', '195', '196', '197', '198', '199'], 'layoutname': 'layout4-clusters', 'labe

### sending from jupyter-client to others (e.g. webclient)

In [3]:
# basic example 
# sio.emit('ex', {
#     'usr': 'jupyter-client',
#     'msg': 'Node: 20', 'id': None, 'val': '20', 'fn': 'node'
# }, namespace='/main')


### change project 

In [4]:
# get all projects in backend 

import GlobalData as GD

# print as pretty table with index from  0 - x
allprojects = []
for i, proj in enumerate(GD.listProjects()):
    allprojects.append((i,proj))
allprojects

[(0, 'CDK5'),
 (1, 'CircLadderGraph-xsmall'),
 (2, 'diffusion'),
 (3, 'GenExpression_01'),
 (4, 'GenExpression_02'),
 (5, 'imunet_250130-X0-inter'),
 (6, 'imunet_250130-X1-inter'),
 (7, 'Interactive_Project_T01'),
 (8, 'JSON_autocore'),
 (9, 'JSON_barbellgraph'),
 (10, 'JSON_Zachary'),
 (11, 'Sphere_Torus'),
 (12, 'Teapot'),
 (13, 'Test'),
 (14, 'TheMandelbulb_edges')]

In [6]:
# choose a project id from list above to change project 
sel_index = 11

sel_id = allprojects[sel_index][0]
sel_name = allprojects[sel_index][1]

sio.emit('ex', {
    'usr': 'jupyter-client',
    'fn': 'dropdown',
    'val': sel_id,
    'msg' : sel_name,
    'id': 'projDD',
}, namespace='/main')

### receive messages from server or web/VR client

In [8]:
# SEND DATA - select a node via jupyter client
sio.emit('ex', {
    'usr': 'jupyter-client',
    'fn': 'node',
    'val': '20',
    'msg' : 'Node: ',
    'id': None,
}, namespace='/main')

In [10]:
# SELECT SOMETHING on Web/VR client and check if received here on jupyter client : 
print("📦 Latest data received:", latest_data)


📦 Latest data received: {'fn': 'plotly2js', 'parent': 'plotly2js', 'val': '{"data": [{"hoverinfo": "none", "line": {"color": "#888", "width": 0.5}, "mode": "lines", "x": [-0.17783944459852113, -0.036143181818314045, null, -0.17783944459852113, -0.9774075952885172, null, -0.17783944459852113, 0.03771591556650938, null, -0.17783944459852113, 0.1785320243543765, null, -0.17783944459852113, 0.9751422817844664, null, -0.036143181818314045, 0.1785320243543765, null, -0.9774075952885172, 0.1785320243543765, null, 0.03771591556650938, 0.1785320243543765, null, 0.1785320243543765, 0.9751422817844664, null], "y": [-0.1381751324195242, -1.0, null, -0.1381751324195242, 0.21123710827831865, null, -0.1381751324195242, 0.9942694081056708, null, -0.1381751324195242, 0.1380462680630298, null, -0.1381751324195242, -0.20537765202749428, null, -1.0, 0.1380462680630298, null, 0.21123710827831865, 0.1380462680630298, null, 0.9942694081056708, 0.1380462680630298, null, 0.1380462680630298, -0.2053776520274942

### change layout (realtime calculation)